In [0]:
# ============================================================
# Silver — Source 07: ShipStation Shipments
#
# Transformations:
#   - Cast all ISO timestamp strings to timestamp
#   - Normalise carrier, service to title case
#   - delivered_at null is valid (not yet delivered)
#   - Reject null shipment_id or order_id → quarantine
#   - Deduplicate on shipment_id
#
# Source:  bronze.src_07_shipments.shipments
# Target:  silver.src_07_shipments.shipments
# Quarantine: silver.quarantine.src_07_shipments
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
from pyspark.sql.window import Window

BRONZE_CATALOG = 'bronze'
SILVER_CATALOG = 'silver'
TARGET_TABLE = f'{SILVER_CATALOG}.src_07_shipments.shipments'
QUARANTINE_TABLE = f'{SILVER_CATALOG}.quarantine.src_07_shipments'

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {SILVER_CATALOG}.src_07_shipments')
print('Silver Source 07 ShipStation — starting...')


In [0]:
# ── LOAD AND CLEAN ────────────────────────────────────────────
bronze = spark.table(f'{BRONZE_CATALOG}.src_07_shipments.shipments')
total = bronze.count()
print(f'Bronze rows: {total}')

# Step 1: Cast timestamps
df = bronze \
    .withColumn('created_at',          F.to_timestamp(F.col('created_at'))) \
    .withColumn('shipped_at',          F.to_timestamp(F.col('shipped_at'))) \
    .withColumn('delivered_at',        F.to_timestamp(F.col('delivered_at'))) \
    .withColumn('estimated_delivery',  F.to_timestamp(F.col('estimated_delivery')))

# Step 2: Normalise
df = df \
    .withColumn('carrier',          F.initcap(F.trim(F.col('carrier')))) \
    .withColumn('service',          F.initcap(F.trim(F.col('service')))) \
    .withColumn('shipping_country', F.upper(F.trim(F.col('shipping_country'))))

# Step 3: Bad rows
bad = df.filter(
    F.col('shipment_id').isNull() |
    F.col('order_id').isNull() |
    F.col('shipped_at').isNull() |
    F.col('carrier').isNull()
).withColumn('quarantine_reason', F.lit('failed_validation')) \
 .withColumn('source_table', F.lit('shipments'))

# Step 4: Good rows
good = df.filter(
    F.col('shipment_id').isNotNull() &
    F.col('order_id').isNotNull() &
    F.col('shipped_at').isNotNull() &
    F.col('carrier').isNotNull()
)

w = Window.partitionBy('shipment_id').orderBy(F.col('shipped_at').desc())
good = good.withColumn('_rn', F.row_number().over(w)) \
           .filter(F.col('_rn') == 1).drop('_rn')

bad_count = bad.count()
good_count = good.count()
print(f'Shipments: {total} total → {good_count} clean, {bad_count} quarantined ({bad_count/total*100:.1f}%)')

# Show carrier distribution
good.groupBy('carrier').count().orderBy('count', ascending=False).show()

# Step 5: Write
if spark.catalog.tableExists(TARGET_TABLE):
    dt = DeltaTable.forName(spark, TARGET_TABLE)
    dt.alias('t').merge(good.alias('s'), 't.shipment_id = s.shipment_id') \
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print('MERGE complete')
else:
    good.write.format('delta').mode('overwrite').saveAsTable(TARGET_TABLE)
    print('Initial load complete')

# Step 6: Quarantine
if bad_count > 0:
    quarantine = bad.select(
        F.lit('src_07_shipments').alias('source'),
        F.col('source_table'),
        F.col('quarantine_reason'),
        F.current_timestamp().alias('quarantined_at'),
        F.to_json(F.struct(*[c for c in bad.columns if c not in ['quarantine_reason','source_table']])).alias('raw_record')
    )
    quarantine.write.format('delta').mode('append') \
        .option('mergeSchema', 'true').saveAsTable(QUARANTINE_TABLE)
    print(f'✅ {bad_count} rows quarantined')
else:
    print('No quarantine rows')


In [0]:
# ── VERIFY ────────────────────────────────────────────────────
count = spark.sql(f'SELECT COUNT(*) as cnt FROM {TARGET_TABLE}').collect()[0]['cnt']
print(f'silver.src_07_shipments.shipments: {count} rows')
spark.sql(f"""
    SELECT carrier, COUNT(*) as cnt,
           SUM(CASE WHEN delivered_at IS NOT NULL THEN 1 ELSE 0 END) as delivered
    FROM {TARGET_TABLE}
    GROUP BY carrier
    ORDER BY cnt DESC
""").show()
